# Tool-Discovery Agent

Discover bioinformatics tools from papers, inject them into your agent, and run tasks.

**Flow:** Scan agent -> Inject tools -> Run tasks

In [ ]:
import os, sys, subprocess
if not os.path.exists('/content/st2/agent_connector'):
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/caixiaoyao2025/st2.git', '/content/st2'], check=True)
ST2_DIR = '/content/st2'
sys.path.insert(0, ST2_DIR)
os.chdir(ST2_DIR)
!pip install -q langchain langchain-openai langgraph pydantic graph-tool-call 2>/dev/null || true
from IPython.display import display, Markdown
display(Markdown('**Setup done**'))

## Step 1 - Input your agent & API key

In [ ]:
import os, subprocess
from IPython.display import display, Markdown
import ipywidgets as widgets

KNOWN_AGENTS = {
    'biomni': 'go',
    'cellagent': 'run',
    'geneagent': 'run',
    'crispr': 'run',
    'biochatter': 'run',
    'langchain': 'invoke',
    'smolagents': 'run',
    'dspy': 'forward',
    'crewai': 'kickoff',
    'metagpt': 'run',
}
PROBE_ORDER = ['go', 'run', 'execute', 'predict', 'forward', 'invoke']

path_input = widgets.Text(
    value='https://github.com/snap-stanford/Biomni.git',
    placeholder='Git URL or /content/my_agent',
    description='Agent:', layout=widgets.Layout(width='90%'))
key_input = widgets.Password(
    value='', placeholder='ark-... or sk-...',
    description='API Key:', layout=widgets.Layout(width='90%'))
model_input = widgets.Text(
    value='deepseek-v4-flash-ga-260731',
    description='Model:', layout=widgets.Layout(width='70%'))
base_input = widgets.Text(
    value='https://ark.cn-beijing.volces.com/api/v3',
    description='Base URL:', layout=widgets.Layout(width='90%'))
btn = widgets.Button(description='Connect agent', button_style='primary')
out = widgets.Output()

def on_connect(_):
    out.clear_output()
    with out:
        agent_src = path_input.value.strip()
        api_key = key_input.value.strip()
        model = model_input.value.strip()
        base_url = base_input.value.strip()
        if not agent_src:
            display(Markdown('**ERROR:** Enter agent path or URL')); return
        if not api_key:
            display(Markdown('**ERROR:** Enter API key')); return
        if agent_src.startswith('http'):
            agent_dir = '/content/_agent_' + agent_src.split('/')[-1].replace('.git', '')
            if not os.path.isdir(agent_dir):
                subprocess.run(['git', 'clone', '--depth', '1', agent_src, agent_dir],
                               capture_output=True, text=True)
            display(Markdown(f'Cloned to `{agent_dir}`'))
        else:
            agent_dir = agent_src
            if not os.path.isdir(agent_dir):
                display(Markdown(f'**ERROR:** `{agent_dir}` not found')); return
        os.environ['OPENAI_API_KEY'] = api_key
        os.environ['OPENAI_BASE_URL'] = base_url
        os.environ['OPENAI_MODEL'] = model
        os.environ['WESTLAKE_API_KEY'] = api_key
        get_ipython().user_ns['agent_dir'] = agent_dir
        get_ipython().user_ns['api_key_val'] = api_key
        get_ipython().user_ns['model_val'] = model
        get_ipython().user_ns['base_url_val'] = base_url
        display(Markdown(f'**Agent:** `{agent_dir}` | **Model:** `{model}` | **Key:** `{api_key[:8]}...`'))

btn.on_click(on_connect)
display(widgets.VBox([path_input, key_input, model_input, base_input, btn, out]))

## Step 2 - Scan agent & detect wiring

In [ ]:
import os, sys, json
from IPython.display import display, Markdown
import yaml

from agent_connector.scanner import build_schema

schema = build_schema(agent_dir, include_evidence=False)
detected = [
    '**Detected:**',
    '- agent_class = `' + str(schema.get('agent_class', 'N/A')) + '`',
    '- registration_method = `' + str(schema.get('registration_method', 'N/A')) + '`',
    '- wiring_style = `' + str(schema.get('wiring_style', 'N/A')) + '`',
]
display(Markdown('<br>'.join(detected)))

reg_path = os.path.join(ST2_DIR, 'data', 'mcp_registry.yaml')
tools = yaml.safe_load(open(reg_path, encoding='utf-8'))['tools']
display(Markdown(f'**Registry:** {len(tools)} tools'))

## Step 3 - Resolve execution method

In [ ]:
from IPython.display import display, Markdown

agent_class = (schema.get('agent_class') or '').lower()
agent_dir_lower = agent_dir.lower()

# Layer 1: known agent -> direct mapping
exec_method = None
for pat, method in KNOWN_AGENTS.items():
    if pat in agent_class or pat in agent_dir_lower:
        exec_method = method
        display(Markdown(f'**Known agent** `{pat}` -> `{method}`'))
        break

# Layer 2: scan source for .go()/.run() hints
if exec_method is None:
    import re
    hits = {}
    for root, dirs, files in os.walk(agent_dir):
        dirs[:] = [d for d in dirs if d not in ('.git','__pycache__','node_modules','.venv')]
        for f in files:
            if f.endswith('.py'):
                try:
                    text = open(os.path.join(root,f), encoding='utf-8', errors='replace').read()
                except: continue
                for m in PROBE_ORDER:
                    hits[m] = hits.get(m, 0) + text.count(f'.{m}(')
    if any(v > 0 for v in hits.values()):
        exec_method = max(hits, key=hits.get)
        display(Markdown(f'**Source hint** -> `{exec_method}` ({hits[exec_method]} occurrences)'))

# Layer 3: default
if exec_method is None:
    exec_method = 'run'
    display(Markdown('**No signal found.** Defaulting to `run`.'))

get_ipython().user_ns['exec_method_override'] = exec_method

## Step 4 - Preflight check

In [ ]:
import os
from IPython.display import display, Markdown
from agent_connector.agent_preflight import preflight, preflight_report

pf = preflight(agent_dir)
display(Markdown(preflight_report(pf)))

In [ ]:
import subprocess, os, sys
from IPython.display import display, Markdown

if pf.status == 'SETUP_REQUIRED':
    pkgs = pf.pip_installable
    if pkgs:
        display(Markdown(f'Installing **{len(pkgs)}** packages...'))
        cmd = [sys.executable, '-m', 'pip', 'install', '-q'] + pkgs
        r = subprocess.run(cmd, capture_output=True, text=True, timeout=300)
        display(Markdown(f'**Done** (returncode={r.returncode})'))
    else:
        display(Markdown('No auto-installable packages'))
else:
    display(Markdown(f'Status = `{pf.status}`'))

## Step 5 - Create agent, inject tools, probe execution

In [ ]:
import os, sys, json, importlib
from IPython.display import display, Markdown

from agent_connector.generator import generate_wiring, load_wrappers, load_adapter

schema['execution_method'] = exec_method_override

# Generate wiring
wiring_dir = os.path.join(ST2_DIR, 'wiring')
wiring = generate_wiring(tools, schema, out_dir=wiring_dir)
display(Markdown('Wiring mode: `' + wiring['mode'] + '`'))

# Load wrappers
sys.path.insert(0, wiring_dir)
sys.path.insert(0, agent_dir)
reg_style = schema.get('registration_style') or 'object'
wrappers = load_wrappers(package_name='generated_tools', registration_style=reg_style)
for w in wrappers:
    if not hasattr(w, 'name'): w.name = getattr(w, '__name__', None)
display(Markdown(f'**{len(wrappers)} wrappers loaded**'))

# Create agent
adapter_cls = load_adapter(schema.get('agent_class') or 'Agent',
                           adapter_path=wiring['artifacts']['adapter'])
agent = None
used_adapter = None

if schema.get('registration_method') or schema.get('agent_class'):
    try:
        used_adapter = adapter_cls()
        agent = used_adapter.create_agent()
        used_adapter.register_tools(agent, wrappers)
        display(Markdown(f'**Agent created:** `{type(agent).__name__}`'))
    except Exception as e:
        display(Markdown(f'`create_agent()` failed: `{e}`'))

if agent is None:
    class DynamicAgent:
        def __init__(self): self.tools = []
    DynamicAgent.add_tool = lambda self, t: self.tools.append(t)
    agent = DynamicAgent()
    for w in wrappers: agent.add_tool(w)
    used_adapter = adapter_cls(agent=agent)
    display(Markdown(f'**Fallback:** DynamicAgent with {len(agent.tools)} tools'))

# Layer 1: known agent -> trust mapping, skip probe
is_known = any(pat in agent_dir.lower() or pat in type(agent).__name__.lower()
               for pat in KNOWN_AGENTS)
if is_known:
    display(Markdown(f'**Known agent** -> using `{exec_method_override}` (no probe needed)'))
else:
    # Layer 2: probe the agent object
    probe = None
    for m in PROBE_ORDER:
        if m == '__call__':
            if callable(agent):
                probe = m; break
        elif hasattr(agent, m) and callable(getattr(agent, m)):
            probe = m; break
    if probe:
        display(Markdown(f'**Probe OK:** `agent.{probe}()` exists'))
        get_ipython().user_ns['exec_method_override'] = probe
    else:
        tried = ', '.join(f'`{m}()`' for m in PROBE_ORDER)
        display(Markdown(f'**Probe FAILED:** tried {tried} -- none found on `{type(agent).__name__}`'))
        display(Markdown('Go to **Step 5b** to provide your agent init code manually.'))

get_ipython().user_ns['agent'] = agent
get_ipython().user_ns['wrappers'] = wrappers
get_ipython().user_ns['adapter'] = used_adapter

## Step 5b - Manual agent init (only if probe failed)

If Step 5 succeeded, **skip this cell**.

If probe failed, write your agent init + tool injection code below.

In [ ]:
from IPython.display import display, Markdown
import ipywidgets as widgets

# Check if Step 5 already succeeded
already_ok = any(pat in type(agent).__name__.lower() or pat in agent_dir.lower()
                 for pat in KNOWN_AGENTS)
if not already_ok:
    for m in PROBE_ORDER:
        if m == '__call__':
            if callable(agent):
                already_ok = True; break
        elif hasattr(agent, m) and callable(getattr(agent, m)):
            already_ok = True; break

if already_ok:
    display(Markdown('Step 5 succeeded. **Skipping.**'))
else:
    display(Markdown('### Agent execution method not recognized\n\n'
        '**Tried:** ' + ', '.join(f'`{m}`' for m in PROBE_ORDER) + '\n\n'
        '**Please paste your agent init code below:**'))
    code_area = widgets.Textarea(
        value='# Example:\n# from my_agent import MyAgent\n# agent = MyAgent(model="gpt-4")\n# agent.add_tools(wrappers)\n',
        placeholder='agent = MyAgent(...)',
        layout=widgets.Layout(width='90%', height='150px'))
    method_input = widgets.Dropdown(
        options=['run', 'execute', 'go', 'predict', 'forward', 'invoke', '__call__'],
        value='run', description='Exec method:', layout=widgets.Layout(width='60%'))
    apply_btn = widgets.Button(description='Apply', button_style='primary')
    out = widgets.Output()

    def on_apply(_):
        out.clear_output()
        with out:
            try:
                local_ns = {'wrappers': wrappers, 'agent_dir': agent_dir}
                exec(code_area.value, local_ns)
                new_agent = local_ns.get('agent')
                if new_agent is None:
                    display(Markdown('Error: no `agent` variable found.'))
                    return
                method = method_input.value
                fn = getattr(new_agent, method, None) if method != '__call__' else new_agent
                if fn is None or not callable(fn):
                    display(Markdown(f'Error: `{method}` not callable.'))
                    return
                get_ipython().user_ns['agent'] = new_agent
                get_ipython().user_ns['exec_method_override'] = method
                agent = new_agent
                display(Markdown(f'**Done!** Agent = `{type(new_agent).__name__}`, method = `{method}`'))
                cfg = {'agent_class': type(new_agent).__name__, 'invoke_method': method, 'init_code': code_area.value}
                cfg_path = os.path.join(agent_dir, '.agent_config.json')
                with open(cfg_path, 'w') as f: json.dump(cfg, f, indent=2)
                display(Markdown(f'Saved to `{cfg_path}` -- next time will skip probing.'))
            except Exception as e:
                display(Markdown(f'Error: `{e}`'))

    apply_btn.on_click(on_apply)
    display(widgets.VBox([code_area, method_input, apply_btn, out]))

## Step 6 - Run tasks via Agent Loop

Ask the agent a bioinformatics question. The LLM will:
1. **Retrieve** relevant tools from the registry
2. **Select** the best tool
3. **Extract** arguments from your query
4. **Execute** the tool
5. **Validate** the result
6. **Replan** if needed

In [ ]:
import os, sys, json
from IPython.display import display, Markdown

from openai import OpenAI
from agent_connector.tool_runner import run_tool_spec, format_result
from agent_connector.tool_spec import make_leaf_spec
from agent_connector.graph_retrieval import build_graph_from_tools, retrieve_tools

client = OpenAI(api_key=os.environ['OPENAI_API_KEY'], base_url=os.environ['OPENAI_BASE_URL'])
MODEL = os.environ['OPENAI_MODEL']

fnmap = {}
for t in tools:
    if t.get('arg_style') == 'subcommand' and t.get('subcommand_details'):
        for sub_name in t['subcommands']:
            leaf = make_leaf_spec(t, sub_name)
            fnmap[leaf['name']] = leaf
    else:
        fnmap[t['name']] = t

def to_fn_schema(spec):
    props = {}
    required = []
    for pname, meta in (spec.get('inputs') or {}).items():
        props[pname] = {'type': 'string', 'description': (meta or {}).get('description', '')}
        if (meta or {}).get('required'):
            required.append(pname)
    return {'type': 'function', 'function': {
        'name': spec['name'], 'description': (spec.get('description') or '')[:300],
        'parameters': {'type': 'object', 'properties': props, 'required': required}}}

schemas = [to_fn_schema(s) for s in fnmap.values()]
graph = build_graph_from_tools(tools)
display(Markdown(f'**Agent loop ready:** {len(fnmap)} tools, model `{MODEL}`'))

In [ ]:
import json, os
from IPython.display import display, Markdown
import ipywidgets as widgets
from openai import OpenAI
from agent_connector.tool_runner import run_tool_spec, format_result
from agent_connector.agent import _select_tool, extract_arguments, validate_arguments, validate_result, MAX_RETRIES
from agent_connector.graph_retrieval import retrieve_tools

client = OpenAI(api_key=os.environ['OPENAI_API_KEY'], base_url=os.environ['OPENAI_BASE_URL'])
MODEL = os.environ['OPENAI_MODEL']

query_input = widgets.Text(
    value='Analyze the FASTA file at /content/sample.fasta',
    placeholder='Ask a bioinformatics question...',
    description='Query:', layout=widgets.Layout(width='95%'))
run_btn = widgets.Button(description='Run agent', button_style='success')
result_out = widgets.Output()

def run_agent_loop(_):
    result_out.clear_output()
    with result_out:
        query = query_input.value.strip()
        if not query:
            display(Markdown('**Enter a query**')); return
        display(Markdown(f'**Query:** {query}'))
        candidates_all = [s['function']['name'] for s in schemas]
        tool_descs = {s['function']['name']: s['function'].get('description', '') for s in schemas}
        excluded = set()
        final_answer = None

        for attempt in range(MAX_RETRIES + 1):
            if graph is not None:
                results = retrieve_tools(graph, query, top_k=5)
                candidate_names = [r[0] for r in results if r[0] not in excluded]
            else:
                candidate_names = [n for n in candidates_all if n not in excluded]
            if not candidate_names:
                candidate_names = [n for n in candidates_all if n not in excluded]
            if not candidate_names:
                display(Markdown('**No candidates**')); break

            tool_name = _select_tool(query, candidate_names, client, MODEL, tool_descs)
            display(Markdown(f'**Turn {attempt+1}:** `{tool_name}`'))
            if tool_name is None:
                display(Markdown('NO_MATCHING_TOOL')); break

            spec = fnmap.get(tool_name)
            if spec is None:
                excluded.add(tool_name); continue

            args = extract_arguments(query, spec, client, MODEL)
            display(Markdown(f'**Args:** `{json.dumps(args)[:200]}`'))

            is_valid, missing = validate_arguments(spec, args)
            if not is_valid:
                display(Markdown(f'**Missing:** {missing}'))
                final_answer = f'Need: {missing}'
                break

            try:
                raw = run_tool_spec(spec, args, timeout_override=300)
                result_text = format_result(raw)
                exec_ok = raw.get('return_code') == 0 and raw.get('status') == 'ok'
                display(Markdown(f'**Result:** `{result_text[:300]}`'))
            except Exception as e:
                display(Markdown(f'**Error:** `{e}`'))
                excluded.add(tool_name)
                continue

            if not exec_ok:
                excluded.add(tool_name)
                continue

            satisfied, reason = validate_result(query, tool_name, result_text, client, MODEL)
            if satisfied:
                final_answer = result_text
                display(Markdown(f'**VALIDATED** - {reason}'))
                break
            else:
                display(Markdown(f'**Not satisfied:** {reason} - replanning...'))
                excluded.add(tool_name)

        display(Markdown(f'---\n### Final answer\n{final_answer or "(none)"}'))

run_btn.on_click(run_agent_loop)
display(widgets.VBox([query_input, run_btn, result_out]))